## Feature Engineering

In this section we construct linguistic features capturing lexical simplicity and repetition in song lyrics.  
These features are computed using the tokenized lyrics produced in the preprocessing stage.


In [97]:
import numpy as np
import pandas as pd
from collections import Counter

df = pd.read_pickle("lyrics_preprocessed.pkl")


### Simplicity and repetition feature functions

In [98]:
def type_token_ratio(tokens):
    
    if len(tokens) == 0:
        return 0
    
    return len(set(tokens)) / len(tokens)


def avg_word_length(tokens):
    
    if len(tokens) == 0:
        return 0
    
    return np.mean([len(word) for word in tokens])


def repetition_ratio(tokens):
    
    if len(tokens) == 0:
        return 0
    
    counts = Counter(tokens)
    
    most_common_count = counts.most_common(1)[0][1]
    
    return most_common_count / len(tokens)


def max_word_frequency(tokens):
    
    if len(tokens) == 0:
        return 0
    
    counts = Counter(tokens)
    
    return counts.most_common(1)[0][1]


def bigram_repetition_ratio(tokens):
    
    if len(tokens) < 2:
        return 0
    
    bigrams = list(zip(tokens[:-1], tokens[1:]))
    
    counts = Counter(bigrams)
    
    most_common_count = counts.most_common(1)[0][1]
    
    return most_common_count / len(bigrams)



#### Apply functions to the dataset

In [99]:
df["ttr"] = df["tokens"].apply(type_token_ratio)

df["avg_word_length"] = df["tokens"].apply(avg_word_length)

df["repetition_ratio"] = df["tokens"].apply(repetition_ratio)

df["max_word_freq"] = df["tokens"].apply(max_word_frequency)

df["bigram_repetition_ratio"] = df["tokens"].apply(bigram_repetition_ratio)


#### Inspect results

In [100]:
df[[
    "tokens",
    "ttr",
    "avg_word_length",
    "repetition_ratio",
    "max_word_freq",
    "bigram_repetition_ratio"
]].head(20)


,tokens,ttr,avg_word_length,repetition_ratio,max_word_freq,bigram_repetition_ratio
0,"[something, bad, bout, happen, know, feel, com...",0.390805,4.724138,0.160920,14,0.069767
1,"[blatt, livin, enough, blunt, takin, puff, sta...",0.504762,4.809524,0.061905,13,0.038278
2,"[could, enough, could, enough, bend, as, baow,...",0.375940,4.313283,0.080201,32,0.060302
3,"[know, know, wrong, girl, feel, like, need, lo...",0.675214,4.769231,0.048433,17,0.014286
4,"[always, end, every, time, meet, somebody, new...",0.404580,4.534351,0.068702,9,0.046154
5,"[promised, mile, combined, must, change, heart...",0.582822,5.092025,0.042945,7,0.018519
6,"[kissin, hope, caught, u, whether, like, show,...",0.561265,4.343874,0.130435,33,0.107143
7,"[hate, give, satisfaction, asking, castle, bui...",0.532258,4.951613,0.037634,7,0.016216
8,"[fine, dusk, light, tellin, baby, thing, eat, ...",0.635514,4.897196,0.056075,6,0.037736
9,"[intro, drake, frank, ocean, bet, mother, woul...",0.633028,4.954128,0.050459,11,0.032258


In [101]:
df[[
    "ttr",
    "avg_word_length",
    "repetition_ratio",
    "bigram_repetition_ratio"
]].describe()

,ttr,avg_word_length,repetition_ratio,bigram_repetition_ratio
count,790.000000,790.000000,790.000000,790.000000
mean,0.516865,4.804120,0.092531,0.060390
std,0.140096,0.402178,0.063469,0.049681
min,0.084615,3.615385,0.015823,0.004184
25%,0.423897,4.543562,0.049231,0.027682
50%,0.505895,4.757764,0.073869,0.044643
75%,0.615176,5.004573,0.114286,0.076190
max,1.000000,6.342857,0.451220,0.361963


In [102]:
df[[
    "ttr",
    "avg_word_length",
    "repetition_ratio",
    "bigram_repetition_ratio"
]].corr()


,ttr,avg_word_length,repetition_ratio,bigram_repetition_ratio
ttr,1.000000,0.256397,-0.494215,-0.483383
avg_word_length,0.256397,1.000000,-0.082607,-0.113113
repetition_ratio,-0.494215,-0.082607,1.000000,0.884677
bigram_repetition_ratio,-0.483383,-0.113113,0.884677,1.000000


### Sentiment function

In [103]:
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download("vader_lexicon")

sia = SentimentIntensityAnalyzer()

def sentiment_features(text):
    
    scores = sia.polarity_scores(text)
    
    polarity = scores["compound"]
    
    intensity = abs(polarity)
    
    return polarity, intensity



[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/yaz/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


#### Apply sentiment function to dataset

In [104]:
df[["sentiment_polarity", "sentiment_intensity"]] = df["lyrics_clean"].apply(
    lambda x: pd.Series(sentiment_features(x))
)


#### Inspect results

In [105]:
df[[
    "lyrics_preproc_v2",
    "sentiment_polarity",
    "sentiment_intensity"
]].head(20)


,lyrics_preproc_v2,sentiment_polarity,sentiment_intensity
0,something bad bout happen know feel coming mig...,0.9380,0.9380
1,blatt livin enough blunt takin puff startin fe...,-0.9971,0.9971
2,could enough could enough bend as baow let coo...,-0.9993,0.9993
3,know know wrong girl feel like need love need ...,-0.9404,0.9404
4,always end every time meet somebody new like j...,0.9953,0.9953
5,promised mile combined must change heart like ...,0.9899,0.9899
6,kissin hope caught u whether like show show br...,0.9877,0.9877
7,hate give satisfaction asking castle built peo...,-0.9804,0.9804
8,fine dusk light tellin baby thing eat bone dri...,-0.9925,0.9925
9,intro drake frank ocean bet mother would proud...,0.9945,0.9945


In [106]:
df[["sentiment_polarity","sentiment_intensity"]].describe()


,sentiment_polarity,sentiment_intensity
count,790.000000,790.000000
mean,0.186131,0.929601
std,0.923071,0.146614
min,-0.999900,0.051600
25%,-0.972525,0.945075
50%,0.904600,0.987450
75%,0.992300,0.996700
max,0.999900,0.999900


## TF-IDF

In [115]:
# TF-IDF Vectorization

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=8,
    max_df=0.8,
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(df["lyrics_preproc_v2"])
tfidf_matrix.shape



(790, 2230)

In [116]:
# Inspecting TF-IDF features
feature_names = tfidf.get_feature_names_out()
feature_names[:50]

array(['abandoned', 'account', 'ace', 'across', 'act', 'act like',
       'actin', 'actin like', 'action', 'actually', 'add', 'addict',
       'addicted', 'addiction', 'address', 'admit', 'adore', 'afford',
       'afraid', 'age', 'ahead', 'aim', 'air', 'album', 'alive', 'almost',
       'alone', 'along', 'already', 'already know', 'alright', 'always',
       'american', 'angel', 'anger', 'angry', 'another', 'another nigga',
       'answer', 'anxious', 'anybody', 'anymore', 'anyone', 'anything',
       'anyway', 'anywhere', 'ap', 'apart', 'apology', 'ar'], dtype=object)

In [117]:
# Convert TF-IDF matrix to DataFrame for easier analysis

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf.get_feature_names_out()
)

tfidf_df.head(20)

,abandoned,account,ace,across,act,act like,actin,actin like,action,actually,...,yes sir,yet,yo,york,young,young metro,young nigga,yup,zero,zone
0,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000
1,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000
2,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.071256,0.0,0.000000
3,0.0,0.0,0.0,0.00000,0.048000,0.053710,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000
4,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000
5,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000
6,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000
7,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000
8,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.073320,0.0,0.0,0.000000,0.0,0.000000
9,0.0,0.0,0.0,0.00000,0.047584,0.053244,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000


In [118]:
# Select engineered features for scaling and combination with TF-IDF features

engineered_features = df[[
    "ttr",
    "avg_word_length",
    "repetition_ratio",
    "max_word_freq",
    "sentiment_polarity"
]]

# Scale engineered features before combining with TF-IDF features

scaler = StandardScaler()
engineered_scaled = scaler.fit_transform(engineered_features)

# Convert scaled engineered features back to DataFrame for combination
engineered_scaled_df = pd.DataFrame(
    engineered_scaled,
    columns=engineered_features.columns
)

# Combine scaled engineered features with TF-IDF features for modeling
X = pd.concat(
    [
        engineered_scaled_df.reset_index(drop=True),
        tfidf_df.reset_index(drop=True)
    ],
    axis=1
)

X.describe()



,ttr,avg_word_length,repetition_ratio,max_word_freq,sentiment_polarity,abandoned,account,ace,across,act,...,yes sir,yet,yo,york,young,young metro,young nigga,yup,zero,zone
count,7.900000e+02,7.900000e+02,7.900000e+02,7.900000e+02,7.900000e+02,790.000000,790.000000,790.000000,790.000000,790.000000,...,790.000000,790.000000,790.000000,790.000000,790.000000,790.000000,790.000000,790.000000,790.000000,790.000000
mean,4.227280e-16,-5.666353e-16,4.497106e-18,8.544501e-17,-1.236704e-17,0.000879,0.000652,0.000662,0.002786,0.002965,...,0.000779,0.004138,0.002372,0.001775,0.006563,0.001226,0.001123,0.000842,0.001031,0.000955
std,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,1.000634e+00,0.009823,0.006503,0.006912,0.017104,0.014562,...,0.009912,0.034008,0.013417,0.022399,0.025114,0.016954,0.007441,0.009707,0.010237,0.009923
min,-3.087339e+00,-2.957616e+00,-1.209357e+00,-1.208380e+00,-1.285689e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-6.640248e-01,-6.482769e-01,-6.826570e-01,-6.887173e-01,-1.256014e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-7.835276e-02,-1.153370e-01,-2.942241e-01,-3.175298e-01,7.788391e-01,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,7.021834e-01,4.987336e-01,3.429723e-01,2.763702e-01,8.739081e-01,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,3.450782e+00,3.828431e+00,5.654943e+00,5.769945e+00,8.821467e-01,0.170418,0.103403,0.106691,0.201507,0.140527,...,0.240037,0.719240,0.173837,0.544645,0.238066,0.403531,0.112879,0.180442,0.168044,0.182584


In [119]:
# Final feature set ready for modeling
X.head()

,ttr,avg_word_length,repetition_ratio,max_word_freq,sentiment_polarity,abandoned,account,ace,across,act,...,yes sir,yet,yo,york,young,young metro,young nigga,yup,zero,zone
0,-0.900387,-0.198999,1.078184,-0.243292,0.815046,0.0,0.0,0.0,0.0,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
1,-0.086449,0.013444,-0.482846,-0.317530,-1.282654,0.0,0.0,0.0,0.0,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
2,-1.006559,-1.221220,-0.194402,1.092983,-1.285038,0.0,0.0,0.0,0.0,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.071256,0.0,0.0
3,1.131001,-0.086806,-0.695236,-0.020580,-1.221189,0.0,0.0,0.0,0.0,0.048,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
4,-0.801996,-0.671195,-0.375679,-0.614480,0.877160,0.0,0.0,0.0,0.0,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0


In [120]:
# Top TF-IDF terms

import numpy as np

# compute average TF-IDF score for each term
tfidf_means = np.asarray(tfidf_matrix.mean(axis=0)).flatten()

# create dataframe
tfidf_summary = pd.DataFrame({
    "term": tfidf.get_feature_names_out(),
    "avg_tfidf": tfidf_means
})

# sort by importance
top_terms = tfidf_summary.sort_values(by="avg_tfidf", ascending=False)

top_terms.head(30)


,term,avg_tfidf
1091,like,0.052655
995,know,0.042058
783,got,0.041292
1310,nigga,0.036628
1164,love,0.035390
709,get,0.033091
75,baby,0.029905
150,bitch,0.028796
748,go,0.027929
1358,one,0.027882


In [124]:
X.to_csv("features_matrix.csv", index=False)
df[["days_in_top50"]].to_csv("target_variable.csv", index=False)
